In [ ]:
# The bottom library is used to create connection between the docker postgresql database and the code for the chatbot this is test environment 

In [1]:
pip install psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------------------ --------- 2.1/2.7 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 13.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import socket
print(socket.gethostbyname("localhost"))

127.0.0.1


In [ ]:
#This is where the connection is happening with docker before this i had installed postgresql on my windows which I disabled and then this was
#working fine 

In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="chatbot_test",
    user="postgres",
    password="postgres"
)

cursor = conn.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone())

cursor.close()
conn.close()

('PostgreSQL 16.12 (Debian 16.12-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit',)


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import psycopg2
import uuid

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample text
text = "Ideam provides engineering and innovation services."

# Generate embedding
embedding = model.encode(text).tolist()

# Connect to DB
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="chatbot_test",
    user="postgres",
    password="postgres"
)

cursor = conn.cursor()

# Insert into knowledge_base
cursor.execute(
    """
    INSERT INTO knowledge_base (id, content, embedding)
    VALUES (%s, %s, %s)
    """,
    (str(uuid.uuid4()), text, embedding)
)

conn.commit()
cursor.close()
conn.close()

print("Inserted successfully")

Inserted successfully


In [ ]:
### This code here is to check the sql table created 

In [9]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="chatbot_test",
    user="postgres",
    password="postgres"
)

query = """
SELECT * FROM knowledge_base;
"""
#df_full = pd.read_sql("SELECT * FROM knowledge_base;", conn)
#df_full

df = pd.read_sql(query, conn)
conn.close()

df

C:\Users\user\AppData\Local\Temp\ipykernel_8016\3470638943.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,content,embedding,created_at
0,b170470c-8ad6-43d1-98fa-fc018162af81,Ideam provides engineering and innovation serv...,"[-0.0218644,-0.08375201,0.021609046,-0.0477073...",2026-02-25 10:44:09.868022


In [ ]:
## this code below is to check the cosine similarity using pgvector closer to 0 much similar far from 0 less similar 

In [11]:
import psycopg2
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Question to test
query_text = "What services does Ideam provide?"

# Generate embedding
query_embedding = model.encode(query_text).tolist()

# Connect to DB
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="chatbot_test",
    user="postgres",
    password="postgres"
)

cursor = conn.cursor()

# Similarity search
cursor.execute("""
    SELECT content,
           embedding <=> %s::vector AS distance
    FROM knowledge_base
    ORDER BY embedding <=> %s::vector
    LIMIT 3;
""", (query_embedding, query_embedding))

results = cursor.fetchall()
results
cursor.close()
conn.close()

results

[('Ideam provides engineering and innovation services.', 0.18838050400076467)]

In [12]:
def retrieve_top_k(query_text, k=3):
    from sentence_transformers import SentenceTransformer
    import psycopg2

    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = model.encode(query_text).tolist()

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        database="chatbot_test",
        user="postgres",
        password="postgres"
    )

    cursor = conn.cursor()

    cursor.execute("""
        SELECT content,
               embedding <=> %s::vector AS distance
        FROM knowledge_base
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (query_embedding, query_embedding, k))

    results = cursor.fetchall()

    cursor.close()
    conn.close()

    return results

In [13]:
retrieve_top_k("What services does Ideam offer?")

[('Ideam provides engineering and innovation services.', 0.21245839558665447)]

In [ ]:
#### the below code is for testing the file structure we have now 

In [1]:
import sys
import os

# Get project root (one level above notebooks/)
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("Project root added:", project_root)

Project root added: C:\Users\user\chatbot_postgresql_test\app


In [2]:
from app.embeddings.embedder import get_embedding

ModuleNotFoundError: No module named 'app'